In [1]:
# Lesson 12: Machine Learning Pipelines

import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

print("Lesson 12 environment is ready!")

Lesson 12 environment is ready!


In [2]:
# Step 2: Generate a synthetic classification dataset

X, y = make_classification(
    n_samples=2000,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    weights=[0.90, 0.10],
    flip_y=0,
    random_state=42
)

# Convert features into a DataFrame
feature_names = [f"Feature_{i+1}" for i in range(12)]

X = pd.DataFrame(X, columns=feature_names)
y = pd.Series(y, name="Target")

# Examine the dataset
print("Dataset shape:", X.shape)
print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentages:")
print(y.value_counts(normalize=True).mul(100).round(2))

print("\nFirst five observations:")
display(X.head())

Dataset shape: (2000, 12)

Target distribution:
Target
0    1800
1     200
Name: count, dtype: int64

Target percentages:
Target
0    90.0
1    10.0
Name: proportion, dtype: float64

First five observations:


,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,Feature_10,Feature_11,Feature_12
0,2.870137,0.285897,-0.306934,-3.192484,-0.269250,1.184744,2.047108,1.196150,1.464201,0.444616,-0.117597,-0.934609
1,0.211739,2.308033,1.643260,0.077143,-1.798694,0.138193,2.660697,1.201417,-0.544870,-0.127133,0.466960,-1.107461
2,1.065849,1.433961,0.931653,-1.680074,-1.512274,0.720989,1.955715,-1.324833,0.965264,0.211282,-0.610273,1.133688
3,1.382553,1.767113,1.937521,-0.492920,-1.182619,1.834431,2.149530,-0.924712,0.307309,-0.294734,0.325725,1.214884
4,-1.840699,0.410251,3.177776,2.741856,2.275103,0.489962,-0.366627,0.252893,-0.691931,-1.583368,-0.819945,-0.177323


In [3]:
# Step 3: Stratified train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

print("\nTraining class percentages:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting class percentages:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

Training features: (1600, 12)
Testing features: (400, 12)

Training target distribution:
Target
0    1440
1     160
Name: count, dtype: int64

Testing target distribution:
Target
0    360
1     40
Name: count, dtype: int64

Training class percentages:
Target
0    90.0
1    10.0
Name: proportion, dtype: float64

Testing class percentages:
Target
0    90.0
1    10.0
Name: proportion, dtype: float64


In [4]:
# Step 4: Build and train a Machine Learning pipeline

pipeline_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train the complete pipeline
pipeline_lr.fit(X_train, y_train)

# Make predictions on the test set
y_pred = pipeline_lr.predict(X_test)

# Evaluate performance
print("Logistic Regression Pipeline Results")
print("=" * 45)

print(classification_report(
    y_test,
    y_pred,
    digits=3,
    zero_division=0
))

Logistic Regression Pipeline Results
              precision    recall  f1-score   support

           0      0.944     0.986     0.965       360
           1      0.792     0.475     0.594        40

    accuracy                          0.935       400
   macro avg      0.868     0.731     0.779       400
weighted avg      0.929     0.935     0.928       400



In [5]:
# Step 5: Cross-validation of the complete pipeline

from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision"
}

cv_results = cross_validate(
    pipeline_lr,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

summary = pd.DataFrame({
    metric: cv_results[f"test_{metric}"]
    for metric in scoring
})

print("Five-fold cross-validation results:")
display(summary.round(3))

print("\nMean performance:")
print(summary.mean().round(3))

print("\nStandard deviation:")
print(summary.std().round(3))

Five-fold cross-validation results:


,accuracy,precision,recall,f1,roc_auc,average_precision
0,0.953,0.905,0.594,0.717,0.945,0.815
1,0.953,0.870,0.625,0.727,0.942,0.792
2,0.931,0.857,0.375,0.522,0.869,0.681
3,0.925,0.654,0.531,0.586,0.902,0.634
4,0.925,0.786,0.344,0.478,0.915,0.693



Mean performance:
accuracy             0.938
precision            0.814
recall               0.494
f1                   0.606
roc_auc              0.915
average_precision    0.723
dtype: float64

Standard deviation:
accuracy             0.014
precision            0.100
recall               0.128
f1                   0.113
roc_auc              0.031
average_precision    0.077
dtype: float64


In [ ]:
# Step 6: Compare Logistic Regression with and without PCA

from sklearn.decomposition import PCA

pipeline_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=8)),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Cross-validation using the same training data and folds
pca_cv = cross_validate(
    pipeline_pca,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Compare mean scores
comparison = pd.DataFrame({
    "Without PCA": summary.mean(),
    "With PCA": {
        metric: pca_cv[f"test_{metric}"].mean()
        for metric in scoring
    }
})

comparison["Difference"] = (
    comparison["With PCA"] - comparison["Without PCA"]
)

print("Pipeline comparison (5-fold cross-validation):")
display(comparison.round(3))

# Fit PCA pipeline on the full training set
pipeline_pca.fit(X_train, y_train)

variance = pipeline_pca.named_steps[
    "pca"
].explained_variance_ratio_.sum()

print(f"\nVariance retained by 8 PCA components: {variance:.2%}")

In [7]:
# Step 7: Supervised feature selection pipeline

from sklearn.feature_selection import SelectKBest, f_classif

pipeline_selection = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(
        score_func=f_classif,
        k=8
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Cross-validation on training data only
selection_cv = cross_validate(
    pipeline_selection,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Compare all three pipelines
comparison_all = pd.DataFrame({
    "Baseline": summary.mean(),
    "PCA (8)": {
        metric: pca_cv[f"test_{metric}"].mean()
        for metric in scoring
    },
    "SelectKBest (8)": {
        metric: selection_cv[f"test_{metric}"].mean()
        for metric in scoring
    }
})

print("Three-pipeline comparison:")
display(comparison_all.round(3))

# Fit the selection pipeline on all training data
pipeline_selection.fit(X_train, y_train)

# Identify selected features
selected_mask = pipeline_selection.named_steps[
    "selector"
].get_support()

selected_features = X_train.columns[selected_mask]

print("\nSelected features:")
print(selected_features.tolist())

Three-pipeline comparison:


,Baseline,PCA (8),SelectKBest (8)
accuracy,0.938,0.899,0.936
precision,0.814,0.000,0.808
recall,0.494,0.000,0.475
f1,0.606,0.000,0.590
roc_auc,0.915,0.683,0.916
average_precision,0.723,0.188,0.724



Selected features:
['Feature_1', 'Feature_2', 'Feature_3', 'Feature_4', 'Feature_5', 'Feature_9', 'Feature_10', 'Feature_12']


In [8]:
# Step 9: Hyperparameter tuning inside a pipeline

from sklearn.model_selection import GridSearchCV

param_grid = {
    "selector__k": [6, 8, 10, 12],
    "model__C": [0.01, 0.1, 1, 10]
}

grid_search = GridSearchCV(
    estimator=pipeline_selection,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True
)

# Fit using training data only
grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation F1-score:")
print(round(grid_search.best_score_, 3))

# Display the five highest-ranked combinations
results = pd.DataFrame(grid_search.cv_results_)

top_results = results[
    [
        "param_selector__k",
        "param_model__C",
        "mean_test_score",
        "std_test_score"
    ]
].sort_values(
    by="mean_test_score",
    ascending=False
).head(5)

print("\nTop 5 parameter combinations:")
display(top_results.round(3))

Best parameters:
{'model__C': 10, 'selector__k': 12}

Best cross-validation F1-score:
0.608

Top 5 parameter combinations:


,param_selector__k,param_model__C,mean_test_score,std_test_score
15,12,10.0,0.608,0.103
11,12,1.0,0.606,0.101
14,10,10.0,0.599,0.107
12,6,10.0,0.597,0.101
10,10,1.0,0.597,0.108


In [9]:
# Step 10: Evaluate the tuned pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Retrieve the tuned pipeline
best_pipeline = grid_search.best_estimator_

# Predictions from both pipelines
models = {
    "Baseline": pipeline_lr,
    "Tuned": best_pipeline
}

evaluation = []

for name, model in models.items():
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    evaluation.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "Average Precision": average_precision_score(
            y_test, probabilities
        )
    })

evaluation_df = pd.DataFrame(evaluation).set_index("Model")

print("Held-out test set comparison:")
display(evaluation_df.round(3))

print("\nTuned model confusion matrix:")
print(confusion_matrix(
    y_test,
    best_pipeline.predict(X_test)
))

Held-out test set comparison:


,Accuracy,Precision,Recall,F1,ROC-AUC,Average Precision
Model,,,,,,
Baseline,0.935,0.792,0.475,0.594,0.912,0.683
Tuned,0.932,0.760,0.475,0.585,0.912,0.683



Tuned model confusion matrix:
[[354   6]
 [ 21  19]]


In [ ]:
# Step 11: Save and reload the trained ML pipeline

import joblib
from pathlib import Path
import numpy as np

# Create a directory for saved models
model_dir = Path("saved_models")
model_dir.mkdir(exist_ok=True)

model_path = model_dir / "lesson_12_logistic_pipeline.joblib"

# Save the complete baseline pipeline
joblib.dump(pipeline_lr, model_path)

print("Pipeline saved successfully!")
print("Location:", model_path.resolve())

# Reload the pipeline
loaded_pipeline = joblib.load(model_path)

# Verify that predictions are identical
original_predictions = pipeline_lr.predict(X_test)
loaded_predictions = loaded_pipeline.predict(X_test)

print("\nPredictions identical:",
      np.array_equal(original_predictions, loaded_predictions))

print("Loaded pipeline steps:")
print(list(loaded_pipeline.named_steps.keys()))

In [11]:
# Step 12: Make predictions with the saved pipeline

# Select five observations
new_data = X_test.iloc[:5].copy()

# Predict classes and probabilities
predicted_classes = loaded_pipeline.predict(new_data)

predicted_probabilities = loaded_pipeline.predict_proba(
    new_data
)[:, 1]

# Compare predictions with actual labels
prediction_results = pd.DataFrame({
    "Actual_Class": y_test.iloc[:5].to_numpy(),
    "Predicted_Class": predicted_classes,
    "Probability_Class_1": predicted_probabilities.round(3)
})

prediction_results["Correct_Prediction"] = (
    prediction_results["Actual_Class"]
    == prediction_results["Predicted_Class"]
)

print("Predictions using the saved ML pipeline:")
display(prediction_results)

Predictions using the saved ML pipeline:


,Actual_Class,Predicted_Class,Probability_Class_1,Correct_Prediction
0,0,0,0.156,True
1,0,0,0.043,True
2,0,0,0.001,True
3,0,1,0.609,False
4,0,0,0.033,True
